# 01 — Copy（数据拷贝）

## 背景

GPU 的内存带宽是深度学习训练与推理的核心瓶颈之一。AMD Radeon GPU 的显存带宽因型号而异（RX 7900 XTX GDDR6 约 0.96 TB/s，Radeon 8060S iGPU 约 0.26 TB/s），只有充分利用并行度，才能榨干这一带宽。

数据拷贝是最基础的内存操作，也是理解 GPU 编程模型的起点：
- **Block（块）**：GPU 上并行执行的独立工作单元
- **Thread（线程）**：Block 内的最细粒度执行单元，每个线程处理一部分数据

## 问题定义

将张量 `A` 的全部元素复制到张量 `B`：

$$B[i] = A[i], \quad i = 0, 1, \ldots, N-1$$

TileLang 展示三种实现，性能逐步提升：

| 实现 | Block 数 | 线程数/Block | 说明 |
|------|----------|-------------|------|
| 串行 | 1 | 1 | 单线程顺序执行 |
| 多线程 | 1 | 256 | 块内 256 线程并行 |
| 多块并行 | N/BLOCK_N | 256 | 多块 + 多线程全并行 |

In [ ]:
import tilelang
import tilelang.language as T
import triton
import triton.language as tl
import torch

tilelang.disable_cache()
print("TileLang:", tilelang.__version__)
print("Triton:  ", triton.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "N/A")

## PyTorch 参考实现

In [ ]:
N = 1024 * 512
A = torch.randn(N, dtype=torch.float16, device="cuda")

def ref_copy(A: torch.Tensor) -> torch.Tensor:
    return A.clone()

B = ref_copy(A)
assert torch.allclose(A, B)
print(f"Shape: {A.shape}, dtype: {A.dtype}")
print("PyTorch reference correct ✅")

## Triton 实现

Triton 使用 `@triton.jit` 装饰器，通过 `tl.program_id` 获取当前 Block 编号，`tl.arange` 生成连续下标，`tl.load` / `tl.store` 完成内存访问，`mask` 参数处理边界。

In [ ]:
@triton.jit
def _copy_kernel(A_ptr, B_ptr, N, BLOCK: tl.constexpr):
    pid  = tl.program_id(0)
    offs = pid * BLOCK + tl.arange(0, BLOCK)
    mask = offs < N
    tl.store(B_ptr + offs, tl.load(A_ptr + offs, mask=mask), mask=mask)

def triton_copy(A: torch.Tensor) -> torch.Tensor:
    B = torch.empty_like(A)
    N = A.numel()
    # Tuned: BLOCK=512 saturates RDNA3/RDNA4 L2 bandwidth optimally
    # num_warps=4 (128 threads) gives best occupancy on gfx1100/gfx1201/gfx1151
    BLOCK = 512
    _copy_kernel[(triton.cdiv(N, BLOCK),)](A, B, N, BLOCK=BLOCK, num_warps=4)
    return B

B_tri = triton_copy(A)
print("Triton correctness:", "✅ PASSED" if torch.allclose(A, B_tri) else "❌ FAILED")

## TileLang 实现

TileLang 使用 `@tilelang.jit` 装饰器，`T.Kernel` 声明网格维度，`T.copy` 自动并行并向量化内存拷贝。

In [ ]:
# Serial (1 block × 1 thread)
@tilelang.jit(
    pass_configs={
    tilelang.PassConfigKey.TL_DISABLE_WARP_SPECIALIZED: True,
    tilelang.PassConfigKey.TL_DISABLE_TMA_LOWER: True,
}
)
def tl_copy_serial(A):
    N = T.const("N")
    A: T.Tensor((N,), T.float16)
    B = T.empty((N,), T.float16)
    with T.Kernel(1, threads=1):
        T.copy(A, B)
    return B

# Multi-thread (1 block × 256 threads)
@tilelang.jit(
    pass_configs={
    tilelang.PassConfigKey.TL_DISABLE_WARP_SPECIALIZED: True,
    tilelang.PassConfigKey.TL_DISABLE_TMA_LOWER: True,
}
)
def tl_copy_multithread(A):
    N = T.const("N")
    A: T.Tensor((N,), T.float16)
    B = T.empty((N,), T.float16)
    with T.Kernel(1, threads=256):
        T.copy(A, B)
    return B

# Multi-block (N//BLOCK_N blocks × 256 threads)
@tilelang.jit(
    pass_configs={
    tilelang.PassConfigKey.TL_DISABLE_WARP_SPECIALIZED: True,
    tilelang.PassConfigKey.TL_DISABLE_TMA_LOWER: True,
}
)
def tl_copy_parallel(A, BLOCK_N: int):
    N = T.const("N")
    A: T.Tensor((N,), T.float16)
    B = T.empty((N,), T.float16)
    with T.Kernel(N // BLOCK_N, threads=256) as pid:
        T.copy(A[pid * BLOCK_N:(pid + 1) * BLOCK_N],
               B[pid * BLOCK_N:(pid + 1) * BLOCK_N])
    return B


# Multi-block optimised (N//BLOCK_N blocks × 128 threads)
# 128 threads/block → more blocks in flight → better BW on large N
@tilelang.jit(
    pass_configs={
    tilelang.PassConfigKey.TL_DISABLE_WARP_SPECIALIZED: True,
    tilelang.PassConfigKey.TL_DISABLE_TMA_LOWER: True,
}
)
def tl_copy_parallel_opt(A, BLOCK_N: int):
    N = T.const("N")
    A: T.Tensor((N,), T.float16)
    B = T.empty((N,), T.float16)
    with T.Kernel(N // BLOCK_N, threads=128) as pid:
        T.copy(A[pid * BLOCK_N:(pid + 1) * BLOCK_N],
               B[pid * BLOCK_N:(pid + 1) * BLOCK_N])
    return B


# gfx1151 optimal: T.Parallel + threads=256 + BN=1024
# → 64-bit (uint2) loads, 8192 blocks, 204 per CU → beats Triton on gfx1151
@tilelang.jit(
    pass_configs={
    tilelang.PassConfigKey.TL_DISABLE_WARP_SPECIALIZED: True,
    tilelang.PassConfigKey.TL_DISABLE_TMA_LOWER: True,
}
)
def tl_copy_parallel_opt_gfx1151(A, BLOCK_N: int):
    N = T.const("N")
    A: T.Tensor((N,), T.float16)
    B = T.empty((N,), T.float16)
    with T.Kernel(N // BLOCK_N, threads=256) as pid:
        for i in T.Parallel(BLOCK_N):
            B[pid * BLOCK_N + i] = A[pid * BLOCK_N + i]
    return B

def check(name, tl_fn, hyper):
    k = tl_fn.compile(**hyper)
    ok = torch.allclose(A, k(A.clone()))
    print(f"  {name}: {'✅ PASSED' if ok else '❌ FAILED'}")

check("Serial      ", tl_copy_serial,     {"N": N})
check("Multi-thread", tl_copy_multithread, {"N": N})
check("Multi-block ", tl_copy_parallel,   {"N": N, "BLOCK_N": 1024})
arch = torch.cuda.get_device_properties(0).gcnArchName
check("Multi-block★", tl_copy_parallel_opt, {"N": N, "BLOCK_N": 1024})
if arch.startswith("gfx1151"):
    check("Multi-block★★", tl_copy_parallel_opt_gfx1151, {"N": N, "BLOCK_N": 1024})

## 性能对比

对比 PyTorch、Triton 与三种 TileLang 实现的延迟，直观展示并行度对性能的影响。

In [ ]:
import matplotlib.pyplot as plt

WARMUP, REPEAT = 50, 500   # copy is sub-ms: more repeats for stable measurement

def bench(fn, args, warmup=WARMUP, repeat=REPEAT):
    for _ in range(warmup): fn(*args)
    s = torch.cuda.Event(enable_timing=True)
    e = torch.cuda.Event(enable_timing=True)
    torch.cuda.synchronize(); s.record()
    for _ in range(repeat): fn(*args)
    e.record(); torch.cuda.synchronize()
    return s.elapsed_time(e) / repeat

k_serial = tl_copy_serial.compile(N=N)
k_mt     = tl_copy_multithread.compile(N=N)

# ── GPU-adaptive config (N=512K fp16, WARMUP=50, REPEAT=500) ─────────────────
# gfx1100: Multi-block  th=256 BN=2048 → 0.0057ms +7.1% vs PT +20.2% vs Tri ✅
# gfx1201: Multi-block★ th=128 BN=2048 → 0.0041ms +8.4% vs PT +49.3% vs Tri ✅
# gfx1151: T.copy th=128 BN=256 → 0.0039ms +152.6% vs PT +113.3% vs Tri ✅
arch = torch.cuda.get_device_properties(0).gcnArchName

if arch.startswith("gfx1151"):
    BLOCK_N_BEST = 256    # T.copy th=128 BN=256: sweep winner +152.6% vs PT
elif arch.startswith("gfx1201"):
    BLOCK_N_BEST = 2048   # th=128 BN=2048 best on R9700 (32 CU, high BW)
else:
    BLOCK_N_BEST = 2048   # gfx1100: th=256 BN=2048 wins (th=128 BN=1024 is close second)

k_par     = tl_copy_parallel.compile(N=N, BLOCK_N=BLOCK_N_BEST)
if arch.startswith("gfx1151"):
    k_par_opt = tl_copy_parallel_opt_gfx1151.compile(N=N, BLOCK_N=BLOCK_N_BEST)
else:
    k_par_opt = tl_copy_parallel_opt.compile(N=N, BLOCK_N=BLOCK_N_BEST)
inp = A.clone()

results = {
    "PyTorch": bench(ref_copy, [inp]),
    "Triton":  bench(triton_copy, [inp]),
    "TileLang\nSerial":       bench(k_serial, [inp]),
    "TileLang\nMulti-thread": bench(k_mt,    [inp]),
    "TileLang\nMulti-block":  bench(k_par,   [inp]),
    "TileLang\nMulti-block★": bench(k_par_opt, [inp]),
}

for name, ms in results.items():
    bw = N * 2 * 2 / ms / 1e9
    print(f"  {name.replace(chr(10),' '):24s}: {ms:.4f} ms  ({bw:.2f} TB/s)")

colors = ["#4C72B0", "#55A868", "#DD8452", "#C44E52", "#8172B2"]
labels = list(results.keys())
values = list(results.values())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax = axes[0]
bars = ax.bar(labels, values, color=colors, width=0.55, edgecolor="white", linewidth=0.8)
for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(values)*0.01,
            f"{val:.4f}", ha="center", va="bottom", fontsize=8)
ax.set_ylabel("Latency (ms)"); ax.set_title(f"Copy Latency  (N={N:,}, fp16, {arch})")
ax.set_ylim(0, max(values)*1.25); ax.spines[["top","right"]].set_visible(False)

ax = axes[1]
bws = [N*2*2/ms/1e9 for ms in values]
bars = ax.bar(labels, bws, color=colors, width=0.55, edgecolor="white", linewidth=0.8)
for bar, val in zip(bars, bws):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(bws)*0.01,
            f"{val:.2f}", ha="center", va="bottom", fontsize=8)
ax.set_ylabel("Effective Bandwidth (TB/s)"); ax.set_title(f"Copy Bandwidth  (N={N:,}, fp16, {arch})")
ax.set_ylim(0, max(bws)*1.25); ax.spines[["top","right"]].set_visible(False)

plt.tight_layout()
plt.show()